# Week 2 — Data Wrangling & Processing
**Financial Data Analysis Internship | AAPL**

This notebook implements the Week 2 cleaning and preparation workflow using the Week 1 `data/AAPL.csv` file.

**Important source-structure note:** the repository CSV contains a three-line yfinance header structure. The repository file has 1,261 physical lines, including 3 metadata/header lines and 1,258 dated market records. The notebook validates the actual loaded row count instead of assuming the Week 1 figure.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/AAPL.csv")
PROCESSED_DIR = Path("../data/processed")
REPORT_DIR = Path("../reports")
FIGURE_DIR = Path("../figures/week2")

for p in [PROCESSED_DIR, REPORT_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Data path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())


## 1. Import the raw dataset
The source file has three metadata/header lines. We skip them and assign stable analysis column names.

In [ ]:
df = pd.read_csv(
    DATA_PATH,
    skiprows=3,
    names=["Date", "Close", "High", "Low", "Open", "Volume"]
)

print("Shape:", df.shape)
display(df.head())


## 2. Initial structural assessment

In [ ]:
print("Columns:", df.columns.tolist())
print("\nData types before conversion:")
print(df.dtypes)

print("\nMissing values before conversion:")
display(df.isna().sum().to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())


## 3. Type conversion and chronological ordering

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

numeric_cols = ["Open", "High", "Low", "Close", "Volume"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.sort_values("Date").reset_index(drop=True)

print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Rows:", len(df))
print(df.dtypes)


## 4. Missing-value audit

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)

quality_missing = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
})

display(quality_missing)


## 5. Duplicate and date-quality audit

In [ ]:
exact_duplicates = int(df.duplicated().sum())
duplicate_date_rows = int(df["Date"].duplicated(keep=False).sum())
invalid_dates = int(df["Date"].isna().sum())

quality_duplicates = pd.DataFrame({
    "check": ["exact duplicate rows", "rows involved in duplicate dates", "invalid/missing dates"],
    "count": [exact_duplicates, duplicate_date_rows, invalid_dates]
})

display(quality_duplicates)


## 6. Financial-domain validation

In [ ]:
negative_mask = (df[numeric_cols] < 0).any(axis=1)

ohlc_inconsistent = (
    (df["High"] < df["Low"]) |
    (df["High"] < df[["Open", "Close"]].max(axis=1)) |
    (df["Low"] > df[["Open", "Close"]].min(axis=1))
)

validation = pd.DataFrame({
    "check": [
        "rows with negative OHLCV values",
        "rows with OHLC consistency violations"
    ],
    "count": [
        int(negative_mask.sum()),
        int(ohlc_inconsistent.sum())
    ]
})

display(validation)


## 7. Conservative cleaning decision

In [ ]:
# Keep the raw file unchanged.
# Remove rows only when they cannot support the analysis after quality issues are documented.
# Do not forward-fill OHLC prices automatically.

clean = df.copy()

# Drop rows with invalid dates because a time-series observation cannot be indexed reliably without a date.
clean = clean.dropna(subset=["Date"])

# For the core financial calculations, a complete OHLC record is required.
# Any rows removed here should be reported first.
core_missing_before = clean[numeric_cols].isna().any(axis=1).sum()
print("Rows with any missing OHLCV value before core-data filtering:", int(core_missing_before))

clean = clean.dropna(subset=numeric_cols).copy()
clean = clean.drop_duplicates().copy()

# If duplicate dates remain, report them instead of silently resolving conflicts.
print("Rows after documented basic cleaning:", len(clean))
print("Duplicate dates remaining:", int(clean["Date"].duplicated().sum()))


## 8. Feature engineering

In [ ]:
clean["Daily_Return"] = clean["Close"].pct_change()
clean["Log_Return"] = np.log(clean["Close"] / clean["Close"].shift(1))
clean["Intraday_Range_Pct"] = (clean["High"] - clean["Low"]) / clean["Close"] * 100
clean["Volume_Change_Pct"] = clean["Volume"].pct_change()
clean["Rolling_Volatility_20D"] = clean["Log_Return"].rolling(20).std() * np.sqrt(252)
clean["Year"] = clean["Date"].dt.year

clean["Log_Volume"] = np.log1p(clean["Volume"])

clean = clean.set_index("Date").sort_index()

display(clean.head())


## 9. Return-based outlier flags

In [ ]:
q1 = clean["Daily_Return"].quantile(0.25)
q3 = clean["Daily_Return"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

clean["Return_Outlier_IQR"] = (
    (clean["Daily_Return"] < lower) |
    (clean["Daily_Return"] > upper)
)

mean_ret = clean["Daily_Return"].mean()
std_ret = clean["Daily_Return"].std()

clean["Return_ZScore"] = (clean["Daily_Return"] - mean_ret) / std_ret
clean["Return_Outlier_Z"] = clean["Return_ZScore"].abs() >= 3

print("IQR lower bound:", lower)
print("IQR upper bound:", upper)
print("IQR flagged rows:", int(clean["Return_Outlier_IQR"].sum()))
print("Z-score flagged rows:", int(clean["Return_Outlier_Z"].sum()))


## 10. Inspect extreme observations before deciding whether to remove them

In [ ]:
extreme = clean.loc[
    clean["Return_Outlier_IQR"] | clean["Return_Outlier_Z"],
    ["Close", "Daily_Return", "Intraday_Range_Pct", "Volume",
     "Rolling_Volatility_20D", "Return_ZScore"]
].sort_values("Daily_Return")

display(extreme.head(10))
display(extreme.tail(10))


## 11. Save processed data and quality summary

In [ ]:
processed_path = PROCESSED_DIR / "AAPL_cleaned.csv"
quality_path = REPORT_DIR / "week2_quality_summary.csv"

clean.reset_index().to_csv(processed_path, index=False)

quality_summary = pd.DataFrame({
    "metric": [
        "raw_loaded_rows",
        "raw_columns",
        "missing_dates",
        "exact_duplicate_rows",
        "duplicate_date_rows",
        "negative_value_rows",
        "ohlc_consistency_violations",
        "cleaned_rows",
        "IQR_return_outlier_flags",
        "Z_return_outlier_flags"
    ],
    "value": [
        len(df),
        len(df.columns),
        invalid_dates,
        exact_duplicates,
        duplicate_date_rows,
        int(negative_mask.sum()),
        int(ohlc_inconsistent.sum()),
        len(clean),
        int(clean["Return_Outlier_IQR"].sum()),
        int(clean["Return_Outlier_Z"].sum())
    ]
})

quality_summary.to_csv(quality_path, index=False)

print("Saved:", processed_path)
print("Saved:", quality_path)
display(quality_summary)


## 12. Optional quality visualizations

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(clean.index, clean["Close"])
plt.title("AAPL Close Price After Data Preparation")
plt.xlabel("Date")
plt.ylabel("Close")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "aapl_close_price_week2.png", dpi=150)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(clean.index, clean["Rolling_Volatility_20D"])
plt.title("AAPL 20-Day Rolling Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "aapl_rolling_volatility_week2.png", dpi=150)
plt.show()


## Week 2 conclusion
The processed dataset is now structured for the next stage of financial analysis. The raw CSV is preserved, quality checks are explicit, derived features are reproducible, and unusual returns are flagged rather than automatically deleted. The saved quality summary provides concrete evidence for the Week 2 report.
